# A resource whose open fails is finalized

Construction and acquisition are separate steps. A resource can exist even when opening it fails.

Use the repository Python environment and run the cells in order. This notebook has its own setup; no previous notebook is required.

## 1. Import the implementation

The lifecycle code comes from AOA itself. The example supplies only the resource or external connection used to observe it.

In [ ]:
!pip install -q aoa-action-machine

In [2]:
import asyncio

from aoa.action_machine.resources import BaseController
from aoa.action_machine.graph.core.exclude_graph_model import exclude_graph_model
from aoa.action_machine.resources.base_resource import BaseResource
from aoa.action_machine.resources.connection_scope import connection_scope
from aoa.action_machine.resources.per_call_connection import PerCallConnection

## 2. Define the objects used in this case

These definitions give the case an observable result without introducing unrelated business behavior.

In [3]:
@exclude_graph_model
class TraceController(BaseController):
    """Record lifecycle calls without opening an external connection."""
    def __init__(self, events):
        """Keep the event list supplied by the example."""
        self.events = events
    @classmethod
    def get_wrapper_class(cls):
        """Allow direct use by nested actions."""
        return None
    async def open(self):
        """Record acquisition."""
        self.events.append("open")
    async def close(self):
        """Record release."""
        self.events.append("close")

## 3. Prepare this call

Create the state and declarations used below. Merely declaring a per-call factory does not run it.

In [4]:
@exclude_graph_model
class BrokenController(TraceController):
    """Fail while acquiring a connection."""
    async def open(self):
        """Report failed acquisition."""
        self.events.append("open failed")
        raise OSError("unavailable")
events = []

## 4. Run and check the observation

The assertions test this case’s stated behavior. Printed output is recorded below; expected failures are caught explicitly.

In [5]:
try:
    async with connection_scope({"db": PerCallConnection(
        lambda: BrokenController(events), lambda resource: resource.close(),
    )}):
        raise AssertionError("The body must not run")
except OSError as error:
    print(str(error))
assert events == ["open failed", "close"]
print(events)

unavailable
['open failed', 'close']


## 5. Explain the result

The finalizer must tolerate a partially opened resource. It is registered as soon as the factory returns, before open is awaited.